# `cxr-temporal-multiprior` — Colab quickstart

This notebook clones the repo, installs dependencies, and runs the full
automated test suite (32 smoke + 3 migration tests) on a GPU runtime.
It then demos a `K_max=4` multi-prior forward pass on random images.

**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 is enough; A100 if available).
2. Run the cells in order top-to-bottom.

**What this notebook does NOT do:** real pretraining on MIMIC-CXR. That
requires ~400GB of DICOM JPGs which aren't available in Colab without a
PhysioNet credentialing step + mounting Google Drive. See the very last
section for how to wire that up if you have access.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/nprakash1/cxr-temporal-multiprior.git
%cd cxr-temporal-multiprior

## 2. Install dependencies

`hi-ml-multimodal` brings in the upstream `health_multimodal` package
(BioViL-T encoders). Total install time on a fresh Colab runtime is
about 2–3 minutes.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q hi-ml-multimodal

## 3. Sanity check — environment and GPU

In [ ]:
import torch, sys, platform
print('python      :', platform.python_version())
print('torch       :', torch.__version__)
print('cuda avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device      :', torch.cuda.get_device_name(0))
    print('mem (GiB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 4. Run the 32-test architectural smoke suite

Covers: PE migration math, image encoder backward-compat, full-model
loss + backward, variable-K batching contract, joint self-attention
shape K-invariance, `key_padding_mask` isolation, and the BioViL-T
architecture invariants (`(K+1)·L` token sequence; curr-tokens attend
to every prior; etc.).

In [ ]:
!python biovilt/test_smoke.py

## 5. Run the 3-test checkpoint migration suite

Covers: upstream BioViL-T `(2,1,256) → (5,1,256)` migration shape +
contents; strict-load of migrated state into a fresh `TempCXR(K_max=4)`
with zero missing/unexpected keys; bit-identical behavior at K=1 input
(global / patch L2 diff = `0.000e+00`).

In [ ]:
!python biovilt/test_migration.py

## 6. Demo — build a `K_max=4` multi-prior model and run a forward pass

This shows the public API. We construct `TempCXR` at `K_max=4`,
feed it a batch with a mix of K=0, K=2, K=4 priors (variable-K
batching), and confirm the output shapes are K-invariant: global
`(B, 128)`, patches `(B, 196, 128)`.

In [ ]:
import sys, os
sys.path.insert(0, 'biovilt')
import torch
from tempcxr.modules.tempcxr_model import TempCXR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
K_MAX  = 4

model = TempCXR(mode='biovilt', K_max=K_MAX).to(device).eval()
print(f'TempCXR(K_max={K_MAX}) constructed on {device}')
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  total params: {n_params:.1f}M')

In [ ]:
# A batch of 3 samples with K = 0, 2, 4 priors respectively.
B, C, H, W = 3, 3, 448, 448
torch.manual_seed(0)

curr_imgs  = torch.randn(B, C, H, W,            device=device)
prior_imgs = torch.randn(B, K_MAX, C, H, W,     device=device)
prior_mask = torch.tensor(
    [[False, False, False, False],   # sample 0: K_i = 0
     [True,  True,  False, False],   # sample 1: K_i = 2
     [True,  True,  True,  True ]],  # sample 2: K_i = 4
    dtype=torch.bool, device=device,
)
texts = [
    'No acute cardiopulmonary process.',
    'Stable mild cardiomegaly compared to prior. No focal consolidation.',
    'Worsening bilateral lower lobe opacities concerning for pneumonia.',
]

with torch.no_grad():
    out = model(curr_imgs, prior_imgs, prior_mask, texts=texts)

for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:18s} : {tuple(v.shape)}  dtype={v.dtype}')
    else:
        print(f'  {k:18s} : (non-tensor)')

If you got here cleanly, every piece of the multi-prior pipeline is
working on this Colab runtime: data formatting → joint self-attention
over `(K+1)·L = 980` tokens → K-invariant `(B, 128) / (B, 196, 128)`
outputs, with `key_padding_mask` correctly isolating the K_i=0 sample.

---

## 7. (Optional) Migrate the official BioViL-T checkpoint to `K_max=4`

Downloads the upstream BioViL-T weights to a temp dir (~150MB) and runs
the canonical row-0-copy + replicate-last-row migration to produce a
`type_embed_multi : (5, 1, 256)` table. The migration utility
(`biovilt/migrate_checkpoint.py`) handles three input formats: raw
upstream state, our training-checkpoint envelope, and our own raw
state at a different `K_max`.

In [ ]:
!python biovilt/migrate_checkpoint.py --help

## 8. Running real training on Colab (caveats)

`biovilt/resume_train.py` is a full DDP training script with hard-coded
absolute paths to MIMIC-CXR on the Marlowe cluster:

```
/scratch/m000081/yunhe/dataset/MIMIC-CXR/mimic-cxr-jpg/2.0.0/files
/scratch/m000081/eprakash/temporal/...
```

To run anything resembling real training on Colab you would need to:

1. **Get the data.** MIMIC-CXR-JPG requires PhysioNet credentialed
   access (free but takes ~1 day approval). For a quick smoke train,
   the `subset_out/` CSVs in the repo already define a 500-patient
   split, but they reference the same cluster paths — you'd need to
   rewrite the CSVs' DICOM paths to point at your Drive-mounted copy.
2. **Edit the paths** in `biovilt/resume_train.py` (`IMAGE_ROOT`,
   `CSV_DIR`, `CHECKPOINT_DIR`, `LOG_DIR`) to your Drive locations.
3. **Drop DDP** — Colab only gives you 1 GPU, so replace the
   `setup_ddp()` + `DistributedDataParallel` wrap with a single-device
   path, or just launch with `torchrun --nproc_per_node=1`.
4. **Run** something like:

    ```bash
    !torchrun --nproc_per_node=1 biovilt/resume_train.py \
        --k-max 4 --mode biovilt \
        --init-from /content/drive/MyDrive/biovil_t_image_model_proj_size_128.pt
    ```

For purely architectural validation, sections 4–7 above are exactly
what you need.